# Online vs offline coil maps: the horizontal streaks

Do the online ESPIRiT maps (the measurement's ~15 centre lines × a 32-row readout band, kernel 4)
put a streak-shaped error under every reconstruction? Per slice, with the training protocol:

- **floor PSNR:** |S_onᴴ F⁻¹ k_full| (the best image the online operator can represent) against the
  offline SENSE target. No network can beat it.
- **streak h / v:** variance of the row (column) means of that difference over its variance, in the
  foreground. Near 0 means unstructured error; near 1 means stripes. Horizontal stripes show up as a
  high **h**.
- **target vs RSS:** the same index for |x_off| − RSS. Are the streaks already in the target?

The code is `scripts/compare_online_offline_smaps.py` (the headless version). Needs a GPU.

In [ ]:
%matplotlib inline
import os, pathlib, sys
import pandas as pd
import torch
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if not (ROOT / "physics").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from scripts.compare_online_offline_smaps import setup, compare_slice, _panel

In [ ]:
CONFIG    = "config/knee/guided/tgvn_c8_R8_g1.json"   # the protocol (and its data roots)
SPLIT     = "val"
N_VOLUMES, SLICES_PER_VOLUME = 6, 3
VARIANTS  = ["k6:kernel_size=6", "clamp96:clamp=96", "walsh:method=walsh"]   # on top of the config's
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ds, items, mri, variants, sigma = setup(CONFIG, SPLIT, N_VOLUMES, SLICES_PER_VOLUME, VARIANTS)
gen = torch.Generator(device=DEV).manual_seed(0)
rows, panels = [], []
for idx in items:
    row, panel = compare_slice(ds, idx, mri, variants, sigma, gen, DEV)
    rows.append(row); panels.append(panel)
df = pd.DataFrame(rows).set_index(["volume", "slice"])
print(f"{len(items)} slices, noise {sigma:g}, variants: {[n for n, _ in variants]}")

In [ ]:
names = [n for n, _ in variants]
cols = ["off_vs_rss_streak_h", "off_vs_rss_streak_v"] + \
       [f"{n}_{q}" for n in names for q in ("floor_psnr", "streak_h", "streak_v")]
df[cols].describe().loc[["mean", "50%", "min", "max"]].T.round(3)

**Reading it:** `online_streak_h` well above `online_streak_v`, and above the target's own
`off_vs_rss_streak_h`, points at the online maps. If a variant brings `streak_h` down and
`floor_psnr` up, that variant's setting is the fix.

In [ ]:
SHOW = 3                                              # slices to plot (the first variant, "online")
for (vol, sl), p in list(zip(df.index, panels))[:SHOW]:
    _panel(None, f"{vol} slice {sl}", *p); plt.show()